In [82]:
# 서울시 지하철역 엘리베이터 위치 정보

In [83]:
import requests               # 서울시 Open API 호출 (HTTP 요청용)
import time                   # API 호출 간격 조절 및 대기 시간 생성용
import polars as pl           # Pandas 대용, 고속 데이터프레임 처리 및 파케(Parquet) 저장용
from datetime import datetime # 수집 날짜/시간을 데이터에 기록하기 위한 타임스탬프용
from pathlib import Path      # 로컬 SSD 파일 경로(디렉토리 생성 및 관리) 제어용
import os
from sqlalchemy import create_engine, text
import sys

In [107]:
now = datetime.now()
dt_suffix = now.strftime("%Y%m%d")

## workspace/korea-public-data-pipeline-hub/01_seoul_subway_elevator/src/notebooks
current_dir = Path(".").resolve().as_posix()[3:]

print(current_dir)

workspace/korea-public-data-pipeline-hub/01_seoul_subway_elevator/src/notebooks
workspace/korea-public-data-pipeline-hub/data_lake/seoul_subway
D:\workspace\korea-public-data-pipeline-hub\01_seoul_subway_elevator\src\notebooks
D:\workspace\korea-public-data-pipeline-hub\01_seoul_subway_elevator


In [85]:
API_KEY = "7656714c4567757336356769446c6b" # KEY: 고유 인증키
SERVICE_NAME = "tbTraficElvtr"
START_INDEX = 1                            # START_INDEX: 페이징 시작 행 번호
END_INDEX = 1000                          # END_INDEX: 페이징 종료 행 번호 |
all_rows= []

##  http://openapi.seoul.go.kr:8088/(인증키)/xml/tbTraficElvtr/1/5/

In [86]:
# if str(Path("../").resolve()) not in sys.path:
#     sys.path.append(str(Path("../").resolve()))

# 2️⃣ [홍 대리 맞춤형 팩트 경로] src 폴더가 기준이 되었으니 그 아래 'ods'에서 바로 가져오면 끝!
from ods.SeoulOpenApiClient import SeoulOpenApiClient #

# 3️⃣ 인스턴스 생성 및 가동!
subway_api: SeoulOpenApiClient = SeoulOpenApiClient(API_KEY)
raw_data:dict  = subway_api.fetch_data(
    service_name=SERVICE_NAME,
    start_idx=START_INDEX,  # 💡 중간의 req_type은 건너뛰고 바로 여기로 배달!
    end_idx=END_INDEX
)

In [87]:
subway_status:dict = raw_data.get("tbTraficElvtr")

# 딕셔너리들이 담은 리스트로 변환)
subway_rows:list = subway_status.get("row")

In [88]:
len(subway_rows)

552

In [89]:
df: pl.DataFrame = pl.DataFrame(subway_rows)
print(type(df))

<class 'polars.dataframe.frame.DataFrame'>


In [108]:
df.write_parquet("../../../data_lake/seoul_subway/seoul_subway_elevator.parquet")